# Week 5b — Stress detection on UBFC-Phys: rPPG vs contact BVP vs EDA, leave-subjects-out

**Why UBFC-Phys.** Unlike MCD-rPPG, stress here is **induced and within-person**. 56 participants each have three 3-minute recordings:
- **T1:** rest.
- **T2:** speech task (job-interview style).
- **T3:** arithmetic task (counting down aloud).

Participants were randomly assigned to a **test** (stressful) or **ctrl** (relaxed) version of T2/T3. Contact references are BVP at 64 Hz and EDA at 4 Hz (Empatica E4). The videos are 1024×1024 at 35 fps with **no audio**, so rPPG + audio still needs StressID.

**Question:** can rPPG tell rest from stress tasks for people the model has never seen?

**Design:**
- Labels: T1 = rest (0), T2/T3 = stress (1). Analysed for all subjects, then the test and ctrl groups separately.
- Feature sets:
  - rPPG physiology (HR, HRV).
  - rPPG quality and face motion: a **shortcut control**. People talk in T2/T3, so motion alone could separate the tasks.
  - Contact BVP (ceiling).
  - EDA.
  - BVP + EDA.
- Two settings:
  - **raw:** one recording at a time.
  - **subject-normalised:** each feature z-scored within the person's own three recordings, as in most of the literature.
- Evaluation: **leave-subjects-out** (5 folds × 5 repeats; no person in both train and test). Reports AUC, accuracy, balanced accuracy and F1 with subject-bootstrap 95% CIs, plus **rest-identified rate**: for each person, is the recording with the lowest stress score the rest one? Chance is 33%.
- Transformer on 30 s pulse windows (rPPG, and BVP as ceiling). A GPU is recommended.

**Getting the data: straight from IEEE DataPort into Colab, nothing on your desktop.** Each subject is a ~14 GB zip (`s1.zip` … `s56.zip`, ~790 GB in total). The notebook works **one subject at a time**:
1. Download the zip from IEEE's storage to Colab's local disk (fast; the free Colab disk holds one subject comfortably).
2. Extract the face colour traces from the 3 videos.
3. Save only the small files to **Google Drive**: traces, BVP, EDA, info and anxiety; about 20 MB per subject.
4. Delete the zip and videos.

The full videos are **not** kept in Drive by default: 14 GB per subject would fill a free 15 GB Drive after one subject. If you have a large Drive plan and want the zips kept, set `KEEP_ZIPS_IN_DRIVE = True`.

**One-time login setup (about 2 min).** IEEE only serves the files to a logged-in session, so Colab needs your browser's login cookie:
1. In Chrome, while logged in on the [UBFC-Phys page](https://ieee-dataport.org/open-access/ubfc-phys-2), press **F12** to open DevTools and go to the **Network** tab.
2. Click the `s1.zip` link on the page. In the Network list, click the request whose name starts with `3658?key=...` (from `s3-download-url`).
3. Under **Request Headers**, copy the whole value of **`cookie:`**.
4. You can cancel the download that starts in your browser.
5. In Colab, open **Secrets** (key icon in the left bar) → *Add new secret*. Name it `IEEE_COOKIE`, paste the value, and switch on **Notebook access**.

The cookie is your login, so never paste it into a notebook cell or share it. If downloads later fail with "not logged in", your session expired: copy a fresh cookie and update the secret.

**Other sources still work**, and are checked first: extracted folders `s<N>/` or zips `s<N>.zip` in `MyDrive/UBFC-Phys/raw/` or `/content/ubfc_raw/`, or direct links in `MyDrive/UBFC-Phys/urls.txt` (`s1 https://...`).

**First run:** set `MAX_NEW_SUBJECTS = 2` and check that both subjects finish with `status: 'ok'`. Then set it to 56. The run is resumable: if Colab disconnects, re-run sections 0–1 and finished subjects are skipped. Expect roughly 5–10 min per subject.

## 0. Setup

In [1]:
!pip -q install "pandas==2.2.3" "opencv-python-headless<5"
import cv2, pandas, shutil
print('OpenCV', cv2.__version__, '| pandas', pandas.__version__, '| wget:', shutil.which('wget') is not None)
if int(cv2.__version__.split('.')[0]) >= 5 or not hasattr(cv2, 'CascadeClassifier'):
    print('>>> OpenCV 5 has no Haar face detector: Runtime > Restart session, then run from the top. <<<')

OpenCV 4.14.0 | pandas 2.2.3 | wget: True


In [2]:
import os, re, json, time, warnings, shutil, subprocess, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

from google.colab import userdata
try:
  v = userdata.get('IEEE_COOKIE')
  print('found, length', len(v), '| looks right:', 'SESS' in v or '=' in v)
except Exception as e:
  print(type(e).__name__, e)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

UBFC_ROOT = Path('/content/drive/MyDrive/UBFC-Phys') if IN_COLAB else Path('./UBFC-Phys')
LOCAL_RAW = Path('/content/ubfc_raw') if IN_COLAB else Path('./ubfc_raw')
SRC_DIRS  = [UBFC_ROOT / 'raw', LOCAL_RAW]          # where s<N>/ folders or s<N>.zip may be
URL_FILE  = UBFC_ROOT / 'urls.txt'

# ---- IEEE DataPort (your s1.zip link: .../dataport/s3-download-url/3658?key=<base64 of 'open/49099/s1.zip'>) ----
IEEE_BASE          = 'https://ieee-dataport.org'
IEEE_NODE          = 3658
IEEE_KEY_PREFIX    = 'open/49099/'
KEEP_ZIPS_IN_DRIVE = False      # True keeps each 14 GB zip in MyDrive/UBFC-Phys/raw (needs a large Drive plan)
IEEE_COOKIE = None
try:
    from google.colab import userdata
    IEEE_COOKIE = userdata.get('IEEE_COOKIE')
except Exception:
    IEEE_COOKIE = os.environ.get('IEEE_COOKIE')
print('IEEE login cookie:', 'found' if IEEE_COOKIE else 'NOT set (add the Colab secret IEEE_COOKIE, see the top cell)')
OUT       = UBFC_ROOT / 'processed'
TRACES    = OUT / 'traces'                          # vid_s<N>_T<k>.npz (face colour traces, small)
SMALL     = OUT / 'small'                           # s<N>/ bvp, eda, info, anxiety files
FEAT_DIR  = OUT / 'features'
RES_DIR   = OUT / 'results'
TMP       = Path('/content/ubfc_tmp') if IN_COLAB else Path('./ubfc_tmp')
for p in (UBFC_ROOT, OUT, TRACES, SMALL, FEAT_DIR, RES_DIR, TMP, LOCAL_RAW):
    p.mkdir(parents=True, exist_ok=True)

SUBJECTS          = list(range(1, 57))
MAX_NEW_SUBJECTS  = 11           # subjects to process in this run: 2 for a first test, then 56
N_EXTRACT_WORKERS = 3           # one process per task video; 1 = run in the notebook process
FRAME_SCALE       = 0.5         # 1024x1024 -> 512x512 before face detection / ROI means (faster, same means)
DEFAULT_FPS       = 35.0        # UBFC-Phys camera rate (used if the file header is implausible)
BVP_FS, EDA_FS    = 64.0, 4.0
FS_RPPG           = 30.0
HR_BAND           = (0.7, 3.5)
N_FOLDS, N_REPEATS, N_BOOT = 5, 5, 2000
print('UBFC root:', UBFC_ROOT, '| free local disk: %.0f GB' % (shutil.disk_usage('/').free / 1e9))

found, length 726 | looks right: True
Mounted at /content/drive
IEEE login cookie: found
UBFC root: /content/drive/MyDrive/UBFC-Phys | free local disk: 121 GB


## 1. Obtain each subject → face traces → delete the video

The face-trace extractor is the same as weeks 2–3 (four skin regions: face, forehead and both cheeks; duplicate-frame and face-found flags). It now also stores the **face-box track**, used for the motion shortcut control. Each video is processed in a fresh worker process.

In [3]:
%%writefile /content/ubfc_roi_worker.py
# Face colour traces for one video, run in a fresh ("spawn") process.
import time, shutil
from pathlib import Path
import numpy as np
import cv2

ROI_NAMES = ['face', 'forehead', 'left_cheek', 'right_cheek']
ROI_BOXES = {'face': (0.15, 0.85, 0.10, 0.90), 'forehead': (0.25, 0.75, 0.05, 0.25),
             'left_cheek': (0.12, 0.38, 0.50, 0.75), 'right_cheek': (0.62, 0.88, 0.50, 0.75)}

def cascade_file():
    root = getattr(getattr(cv2, 'data', None), 'haarcascades', None)
    p = Path(root or '.') / 'haarcascade_frontalface_default.xml'
    if not p.exists():
        import urllib.request
        p = Path('/content/haarcascade_frontalface_default.xml')
        if not p.exists():
            urllib.request.urlretrieve('https://raw.githubusercontent.com/opencv/opencv/4.x/data/haarcascades/'
                                       'haarcascade_frontalface_default.xml', p)
    return str(p)

def extract(video_path, out_path, frame_scale=0.5, detect_every=10, ema=0.3, det_scale=0.5):
    cv2.setNumThreads(1)
    det = cv2.CascadeClassifier(cascade_file())
    cap = cv2.VideoCapture(str(video_path))
    fps_c = cap.get(cv2.CAP_PROP_FPS)
    traces, dup, found, boxes, box, prev = [], [], [], [], None, None
    i = 0
    while True:
        ok, bgr = cap.read()
        if not ok:
            break
        if frame_scale != 1:
            bgr = cv2.resize(bgr, None, fx=frame_scale, fy=frame_scale, interpolation=cv2.INTER_AREA)
        small = cv2.resize(bgr, (64, 48), interpolation=cv2.INTER_AREA)
        dup.append(prev is not None and np.array_equal(small, prev)); prev = small
        if i % detect_every == 0:
            g = cv2.resize(cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY), None, fx=det_scale, fy=det_scale)
            faces = det.detectMultiScale(g, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))
            d = None
            if len(faces):
                x, y, w, h = max(faces, key=lambda b: b[2] * b[3])
                d = np.array([x, y, w, h], float) / det_scale
                box = d if box is None else (1 - ema) * box + ema * d
            found.append(d is not None)
        else:
            found.append(found[-1] if found else False)
        row = np.full((len(ROI_NAMES), 3), np.nan, np.float32)
        boxes.append(box if box is not None else np.full(4, np.nan))
        if box is not None:
            x, y, w, h = box
            rgb = bgr[..., ::-1]
            for k, n in enumerate(ROI_NAMES):
                fx0, fx1, fy0, fy1 = ROI_BOXES[n]
                xa, xb = int(max(0, x + fx0 * w)), int(min(bgr.shape[1], x + fx1 * w))
                ya, yb = int(max(0, y + fy0 * h)), int(min(bgr.shape[0], y + fy1 * h))
                if xb - xa < 4 or yb - ya < 4:
                    continue
                patch = rgb[ya:yb, xa:xb]
                ycc = cv2.cvtColor(bgr[ya:yb, xa:xb], cv2.COLOR_BGR2YCrCb)
                skin = (ycc[..., 1] >= 133) & (ycc[..., 1] <= 173) & (ycc[..., 2] >= 77) & (ycc[..., 2] <= 127)
                px = patch[skin] if skin.sum() > 0.2 * skin.size else patch.reshape(-1, 3)
                row[k] = px.mean(0)
        traces.append(row); i += 1
    cap.release()
    np.savez_compressed(out_path, traces=np.array(traces, np.float32), roi_names=np.array(ROI_NAMES),
                        dup=np.array(dup), face_found=np.array(found), fps_container=fps_c,
                        boxes=np.array(boxes, np.float32))
    return i

def run_one(src, out, work_dir, frame_scale=0.5):
    src, out, work = Path(src), Path(out), Path(work_dir)
    work.mkdir(parents=True, exist_ok=True)
    if out.exists():
        return dict(video=src.name, status='skip')
    tmp_out, t0 = work / (src.name + '.npz'), time.time()
    try:
        n = extract(src, tmp_out, frame_scale)
        if n < 300:
            raise RuntimeError(f'only {n} frames decoded')
        shutil.move(str(tmp_out), str(out))
        return dict(video=src.name, status='ok', frames=n, secs=time.time() - t0)
    except Exception as e:
        return dict(video=src.name, status='error', error=repr(e)[:300])
    finally:
        tmp_out.unlink(missing_ok=True)

Writing /content/ubfc_roi_worker.py


In [4]:
# 1a. IEEE DataPort: build each subject's link and check the login works (downloads nothing)
import base64, requests

def ieee_link(n):
    key = base64.b64encode(f'{IEEE_KEY_PREFIX}s{n}.zip'.encode()).decode()
    return f'{IEEE_BASE}/dataport/s3-download-url/{IEEE_NODE}?key={requests.utils.quote(key)}&redirect=1'

def resolve_s3(n):
    # Ask IEEE (with your login cookie) for the temporary direct storage link of s<n>.zip.
    if not IEEE_COOKIE:
        raise RuntimeError('IEEE_COOKIE secret not set')
    h = {'Cookie': IEEE_COOKIE.strip(), 'User-Agent': 'Mozilla/5.0'}
    r = requests.get(ieee_link(n), headers=h, allow_redirects=False, timeout=60)
    loc = r.headers.get('Location', '')
    if r.is_redirect and loc and 'login' not in loc and 'check_logged_in' not in loc:
        return requests.compat.urljoin(r.url, loc)
    m = re.search(r'https?://[^"\'\s<>]+s%d\.zip[^"\'\s<>]*' % n, r.text or '')
    if r.status_code == 200 and m:
        return m.group(0).replace('\\u0026', '&').replace('&amp;', '&')
    raise RuntimeError(f'IEEE did not return a download link for s{n} (HTTP {r.status_code}). '
                       'Your login cookie is probably missing or expired: copy a fresh one into the IEEE_COOKIE secret.')

if IEEE_COOKIE:
    try:
        u = resolve_s3(1)
        r = requests.get(u, headers={'Range': 'bytes=0-0'}, stream=True, timeout=60)   # first byte only
        size = int((r.headers.get('Content-Range', '/0').split('/')[-1]) or 0)
        r.close()
        print(f'login OK -> s1.zip is downloadable ({size / 1e9:.1f} GB)' if r.status_code in (200, 206)
              else f'link found but storage answered HTTP {r.status_code}')
    except Exception as e:
        print('Problem:', e)
else:
    print('No IEEE cookie: only Drive folders/zips and urls.txt can be used.')

login OK -> s1.zip is downloadable (14.8 GB)


In [5]:
import sys, importlib, multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor
sys.path.insert(0, '/content')
import ubfc_roi_worker as W; importlib.reload(W)

NEEDED = re.compile(r'^((vid|bvp|eda)_s\d+_T[123]\.(avi|csv)|info_s\d+\.txt|selfreportedanx_s\d+\.csv)$', re.I)

def read_urls():
    urls = {}
    if URL_FILE.exists():
        for ln in URL_FILE.read_text().splitlines():
            m = re.match(r'\s*s(\d+)\s+(\S+)', ln) or re.search(r'/s(\d+)(?:_zip)?\.zip\S*', ln)
            if m:
                urls[int(m.group(1))] = m.group(2) if m.re.pattern.startswith(r'\s*s') else ln.strip()
    return urls

def find_source(n, urls):
    for d in SRC_DIRS:
        f = d / f's{n}'
        if f.is_dir() and any(f.rglob(f'vid_s{n}_T*.avi')):
            return 'dir', f
        for z in (d / f's{n}.zip', d / f's{n}_zip.zip'):
            if z.exists():
                return 'zip', z
    if n in urls:
        return 'url', urls[n]
    return ('ieee', n) if IEEE_COOKIE else (None, None)

def subject_done(n):
    return all((TRACES / f'vid_s{n}_T{k}.npz').exists() for k in (1, 2, 3)) and \
           any((SMALL / f's{n}').glob('bvp_*T1.csv'))

def download(url, dst):
    free = shutil.disk_usage(dst.parent).free / 1e9
    if free < 35:
        raise RuntimeError(f'only {free:.0f} GB free on the Colab disk; one subject needs ~30 GB')
    t0 = time.time()
    r = subprocess.run(['wget', '-q', '--show-progress', '--progress=bar:force:noscroll', '-c', '--tries=5', '--timeout=60', '-O', str(dst), url])
    if dst.exists():
        print(f'  downloaded {dst.name}: {dst.stat().st_size / 1e9:.1f} GB in {(time.time() - t0) / 60:.1f} min')
    if r.returncode != 0 or not zipfile.is_zipfile(dst):
        raise RuntimeError(f'download failed or not a zip (link expired?): {url[:80]}')
    return dst

def unzip_needed(zp, dest):
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zp) as z:
        for info in z.infolist():
            name = Path(info.filename).name
            if NEEDED.match(name):
                with z.open(info) as fsrc, open(dest / name, 'wb') as fdst:
                    shutil.copyfileobj(fsrc, fdst, 16 * 1024 * 1024)
    return dest

def extract_videos(folder, n):
    jobs = []
    for k in (1, 2, 3):
        v = next(iter(folder.rglob(f'vid_s{n}_T{k}.avi')), None)
        if v is not None:
            jobs.append((str(v), str(TRACES / f'vid_s{n}_T{k}.npz'), str(TMP / 'work'), FRAME_SCALE))
    if N_EXTRACT_WORKERS > 1:
        with ProcessPoolExecutor(min(N_EXTRACT_WORKERS, len(jobs)), mp_context=mp.get_context('spawn')) as ex:
            return list(ex.map(W.run_one, *zip(*jobs)))
    return [W.run_one(*j) for j in jobs]

def process_subject(n, urls):
    if subject_done(n):
        return dict(subject=n, status='skip')
    kind, src = find_source(n, urls)
    if kind is None:
        return dict(subject=n, status='no source')
    t0, work = time.time(), TMP / f's{n}'
    try:
        if kind == 'dir':
            folder = src
        else:
            if kind == 'zip':
                zp = src
            else:
                url = resolve_s3(n) if kind == 'ieee' else src     # fresh temporary link for every subject
                zp = download(url, TMP / f's{n}.zip')
            folder = unzip_needed(zp, work)
            if kind in ('url', 'ieee'):
                if KEEP_ZIPS_IN_DRIVE:
                    (UBFC_ROOT / 'raw').mkdir(exist_ok=True)
                    shutil.move(str(zp), str(UBFC_ROOT / 'raw' / zp.name))
                else:
                    zp.unlink(missing_ok=True)
        (SMALL / f's{n}').mkdir(parents=True, exist_ok=True)
        for f in folder.rglob('*'):
            if f.suffix.lower() in ('.csv', '.txt') and NEEDED.match(f.name):
                shutil.copy(f, SMALL / f's{n}' / f.name)
        res = extract_videos(folder, n)
        bad = [r for r in res if r['status'] == 'error']
        return dict(subject=n, status='error' if bad else 'ok', source=kind, minutes=(time.time() - t0) / 60,
                    error=bad[0]['error'] if bad else None)
    except Exception as e:
        return dict(subject=n, status='error', source=kind, error=repr(e)[:300])
    finally:
        if kind in ('zip', 'url', 'ieee'):
            shutil.rmtree(work, ignore_errors=True)
            (TMP / f's{n}.zip').unlink(missing_ok=True)

urls = read_urls()
print('links in urls.txt:', len(urls), '| IEEE auto-download:', bool(IEEE_COOKIE),
      '| already processed:', sum(subject_done(n) for n in SUBJECTS))
log, new = [], 0
for n in SUBJECTS:
    if new >= MAX_NEW_SUBJECTS:
        break
    if subject_done(n):
        continue
    r = process_subject(n, urls)
    if r['status'] == 'no source':
        continue
    new += 1
    log.append(r)
    print(r)
pd.DataFrame(log).to_csv(OUT / 'extraction_log.csv', mode='a', header=not (OUT / 'extraction_log.csv').exists(), index=False)
done = [n for n in SUBJECTS if subject_done(n)]
print(f'\nsubjects ready: {len(done)} / 56 ->', done)

links in urls.txt: 0 | IEEE auto-download: True | already processed: 56

subjects ready: 56 / 56 -> [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56]


## 2. Features per recording (rPPG, contact BVP, EDA, face motion)

- **rPPG:** POS per face region, SNR-weighted combination, then HR, HRV and beat shape (the same code as weeks 3–4).
- **BVP:** the same features from the 64 Hz wrist BVP, as the ceiling.
- **EDA:** skin-conductance level, slope, and the rate and amplitude of conductance responses.
- **Face motion and signal quality:** face-box movement and rPPG SNR, used as the shortcut control.

In [6]:
from scipy.signal import butter, filtfilt, welch, find_peaks, detrend
_trapz = getattr(np, 'trapezoid', None) or np.trapz

def bandpass(x, fs, lo, hi, order=3):
    b, a = butter(order, [lo / (fs / 2), hi / (fs / 2)], btype='band')
    return filtfilt(b, a, x)

def interp_nans(a):
    # Linear interpolation of NaNs along axis 0 (1-D or 2-D); None if too few valid samples.
    a = np.array(a, float, copy=True)
    flat = a.reshape(len(a), -1)
    idx = np.arange(len(a))
    for j in range(flat.shape[1]):
        m = np.isfinite(flat[:, j])
        if m.sum() < 0.5 * len(m):
            return None
        flat[~m, j] = np.interp(idx[~m], idx[m], flat[m, j])
    return flat.reshape(a.shape)

def pos(rgb, fs, win=1.6):
    # POS (Wang et al. 2017). rgb: (N, 3).
    N = len(rgb)
    l = int(round(win * fs))
    H = np.zeros(N)
    P = np.array([[0, 1, -1], [-2, 1, 1]], float)
    for n in range(l, N + 1):
        C = rgb[n - l:n].T
        mu = C.mean(axis=1, keepdims=True)
        mu[mu == 0] = 1e-9
        S = P @ (C / mu)
        sd1 = S[1].std()
        h = S[0] + (S[0].std() / sd1 if sd1 > 0 else 0) * S[1]
        H[n - l:n] += h - h.mean()
    return H

def spectrum(x, fs, seg_s=12):
    nper = int(min(len(x), seg_s * fs))
    return welch(x, fs, nperseg=nper, noverlap=nper // 2, nfft=max(8192, nper))

def hr_harmonic(x, fs, band=HR_BAND):
    # Harmonic-sum HR (BPM): score(f) = P(f) + 0.5 * P(2f); avoids locking onto the 2nd harmonic.
    f, P = spectrum(x, fs)
    m = (f >= band[0]) & (f <= band[1])
    score = P[m] + 0.5 * np.interp(2 * f[m], f, P, right=0)
    f0 = f[m][np.argmax(score)]
    return 60 * f0, f, P

def snr_db(f, P, f0, half=0.1):
    sig = (np.abs(f - f0) <= half) | (np.abs(f - 2 * f0) <= half)
    rng = (f >= 0.6) & (f <= 4.0)
    num, den = P[sig & rng].sum(), P[~sig & rng].sum()
    return 10 * np.log10(num / den) if num > 0 and den > 0 else np.nan

def detect_beats(x, fs, hr_bpm):
    # Systolic peaks, de-duplicated: min distance 0.6x the expected period.
    period = fs * 60.0 / hr_bpm
    pk, _ = find_peaks(x, distance=max(1, int(0.6 * period)), prominence=0.1 * np.std(x))
    return pk

def hrv_from_peaks(pk, fs, min_dur_freq=100):
    t = pk / fs
    ibi = np.diff(t)
    if len(ibi) < 10:
        return {}
    tb = t[1:]
    med = pd.Series(ibi).rolling(7, center=True, min_periods=1).median().values
    ok = (ibi > 0.33) & (ibi < 1.5) & (np.abs(ibi - med) < 0.2 * med)
    if ok.sum() < 10:
        return dict(ibi_ok_frac=ok.mean())
    ic, tc = ibi[ok], tb[ok]
    d = np.diff(ibi)[ok[1:] & ok[:-1]]
    o = dict(n_beats=len(pk), ibi_ok_frac=ok.mean(), hr_beats=60 / ic.mean(),
             sdnn=1000 * ic.std(ddof=1), rmssd=1000 * np.sqrt(np.mean(d ** 2)) if len(d) else np.nan,
             pnn50=np.mean(np.abs(d) > 0.05) if len(d) else np.nan)
    o['ln_sdnn'] = np.log(max(o['sdnn'], 1e-3))
    o['ln_rmssd'] = np.log(max(o['rmssd'], 1e-3)) if np.isfinite(o['rmssd']) else np.nan
    if tc[-1] - tc[0] >= min_dur_freq and len(ic) >= 60:
        g = np.arange(tc[0], tc[-1], 0.25)
        y = np.interp(g, tc, ic)
        y = y - y.mean()
        f, P = welch(y, 4.0, nperseg=min(256, len(y)))
        lf = _trapz(P[(f >= 0.04) & (f < 0.15)], f[(f >= 0.04) & (f < 0.15)]) * 1e6
        hf = _trapz(P[(f >= 0.15) & (f < 0.40)], f[(f >= 0.15) & (f < 0.40)]) * 1e6
        o.update(lf=lf, hf=hf, ln_lf=np.log(max(lf, 1e-6)), ln_hf=np.log(max(hf, 1e-6)),
                 lf_hf=lf / hf if hf > 0 else np.nan)
    return o

def morphology(x, pk, fs):
    # Average-beat template: rise fraction, half-width, beat-to-template consistency.
    if len(pk) < 10:
        return {}
    T = int(np.median(np.diff(pk)))
    a, b = int(0.6 * T), int(0.4 * T)
    segs = [x[p - a:p + b] for p in pk if p - a >= 0 and p + b <= len(x)]
    if len(segs) < 10:
        return {}
    S = np.array(segs)
    S = (S - S.mean(1, keepdims=True)) / (S.std(1, keepdims=True) + 1e-12)
    tpl = S.mean(0)
    cons = np.mean([np.corrcoef(s, tpl)[0, 1] for s in S])
    i_min = int(np.argmin(tpl[:a + 1]))
    rise = (a - i_min) / T
    half = (tpl.max() + tpl.min()) / 2
    return dict(beat_consistency=cons, rise_frac=rise, half_width=np.mean(tpl > half))

def windowed_hr(x, fs, win=10, step=2):
    n, s = int(win * fs), int(step * fs)
    return np.array([hr_harmonic(x[i:i + n], fs)[0] for i in range(0, len(x) - n + 1, s)])

def pulse_features(x, fs):
    # Shared feature set for any pulse waveform already band-passed (rPPG or contact PPG).
    hr, f, P = hr_harmonic(x, fs)
    o = dict(hr=hr, snr=snr_db(f, P, hr / 60))
    w = windowed_hr(x, fs)
    if len(w):
        o['hr_win_sd'] = np.std(w)
        o['hr_consistency'] = np.mean(np.abs(w - hr) < 5)
    pk = detect_beats(x, fs, hr)
    o.update(hrv_from_peaks(pk, fs))
    o.update(morphology(x, pk, fs))
    return o

In [7]:
from numpy.lib.stride_tricks import sliding_window_view
from scipy.signal import resample_poly

def pos_fast(rgb, fs, win=1.6):
    N, l = len(rgb), int(round(win * fs))
    Wv = sliding_window_view(rgb, l, axis=0)
    mu = Wv.mean(axis=2, keepdims=True); mu[mu == 0] = 1e-9
    C = Wv / mu
    S0, S1 = C[:, 1] - C[:, 2], -2 * C[:, 0] + C[:, 1] + C[:, 2]
    sd1 = S1.std(axis=1, keepdims=True)
    h = S0 + (S0.std(axis=1, keepdims=True) / np.where(sd1 > 0, sd1, np.inf)) * S1
    h -= h.mean(axis=1, keepdims=True)
    H = np.zeros(N)
    for j in range(l):
        H[j:j + len(h)] += h[:, j]
    return H

def load_col(path):
    try:
        x = pd.to_numeric(pd.read_csv(path, header=None).iloc[:, 0], errors='coerce').dropna().values
    except Exception:
        x = np.array([])
    return x.astype(float)

def rppg_from_trace(path):
    z = np.load(path)
    rgb = z['traces'].astype(float); dup = z['dup'].astype(bool); rgb[dup] = np.nan
    fps = float(z['fps_container']); fps = fps if 20 < fps < 61 else DEFAULT_FPS
    t = np.arange(len(rgb)) / fps
    tu = np.arange(0, t[-1], 1 / FS_RPPG)
    sigs, snrs = [], []
    for k in range(rgb.shape[1]):
        c = interp_nans(rgb[:, k, :])
        if c is None:
            continue
        cu = np.column_stack([np.interp(tu, t, c[:, j]) for j in range(3)])
        s = bandpass(pos_fast(cu, FS_RPPG), FS_RPPG, *HR_BAND)
        hr_k, f, Pw = hr_harmonic(s, FS_RPPG)
        snr = snr_db(f, Pw, hr_k / 60)
        sigs.append(s / (np.std(s) + 1e-12)); snrs.append(snr if np.isfinite(snr) else -20)
    o = dict(face_found=float(z['face_found'].mean()), dup_frac=float(dup.mean()), fps=fps, duration=t[-1])
    b = z['boxes'] if 'boxes' in z.files else None
    if b is not None and np.isfinite(b).any():
        bb = pd.DataFrame(b).interpolate(limit_direction='both').values
        cx, cy, w = bb[:, 0] + bb[:, 2] / 2, bb[:, 1] + bb[:, 3] / 2, np.nanmedian(bb[:, 2])
        step = np.hypot(np.diff(cx), np.diff(cy)) / w
        o.update(mot_center_sd=np.hypot(cx.std(), cy.std()) / w, mot_step_mean=step.mean() * fps,
                 mot_step_p95=np.percentile(step, 95) * fps, mot_scale_sd=bb[:, 2].std() / w)
    if not sigs:
        return o, None
    w8 = 10 ** (np.array(snrs) / 10)
    comb = np.sum([wi * si for wi, si in zip(w8, sigs)], axis=0) / w8.sum()
    o.update(pulse_features(comb, FS_RPPG))
    return o, (comb / (comb.std() + 1e-12)).astype(np.float32)

def bvp_from_file(path):
    x = load_col(path)
    if len(x) < 30 * BVP_FS:
        return {}, None
    x = bandpass(detrend(x), BVP_FS, *HR_BAND)
    o = pulse_features(x, BVP_FS)
    y = resample_poly(x, 15, 32)                       # 64 Hz -> 30 Hz for the transformer
    return o, (y / (y.std() + 1e-12)).astype(np.float32)

def eda_features(path, fs=EDA_FS):
    x = load_col(path)
    if len(x) < 30 * fs:
        return {}
    t = np.arange(len(x)) / fs
    b, a = butter(2, 0.05 / (fs / 2), 'low')
    tonic = filtfilt(b, a, x); phasic = x - tonic
    pk, _ = find_peaks(phasic, height=0.01, distance=int(fs))
    return dict(scl_mean=x.mean(), scl_sd=x.std(), scl_slope=np.polyfit(t, x, 1)[0] * 60,
                scr_rate=len(pk) / (t[-1] / 60), scr_amp=float(phasic[pk].mean()) if len(pk) else 0.0,
                phasic_sd=phasic.std())

def subject_info(n):
    d = SMALL / f's{n}'
    info = next(iter(d.glob('info_*.txt')), None)
    toks = info.read_text(errors='ignore').lower().split() if info else []
    out = dict(gender=next((t for t in toks if t in ('m', 'f')), None),
               scenario=next((t for t in toks if t in ('test', 'ctrl')), None))
    anx = next(iter(d.glob('*elf*nx*.csv')), None)
    if anx is not None:
        vals = pd.to_numeric(pd.Series(re.split(r'[,;\s]+', anx.read_text())), errors='coerce').dropna().values
        out.update({f'anx_{i + 1}': v for i, v in enumerate(vals[:6])})
    return out

rows, SIGU = [], {}
for n in tqdm([n for n in SUBJECTS if subject_done(n)]):
    inf = subject_info(n)
    for k in (1, 2, 3):
        r = dict(subject=n, task=k, stress=int(k > 1), **inf)
        try:
            o, s = rppg_from_trace(TRACES / f'vid_s{n}_T{k}.npz')
            r.update({('rppg_' if not kk.startswith(('mot_', 'face_found', 'dup_frac', 'fps', 'duration')) else '') + kk: v
                      for kk, v in o.items()})
            if s is not None:
                SIGU[f'rppg_s{n}_T{k}'] = s
        except Exception as e:
            r['rppg_error'] = repr(e)[:120]
        bp = next(iter((SMALL / f's{n}').glob(f'bvp_s{n}_T{k}.csv')), None)
        if bp is not None:
            o, s = bvp_from_file(bp)
            r.update({'bvp_' + kk: v for kk, v in o.items()})
            if s is not None:
                SIGU[f'bvp_s{n}_T{k}'] = s
        ep = next(iter((SMALL / f's{n}').glob(f'eda_s{n}_T{k}.csv')), None)
        if ep is not None:
            r.update({'eda_' + kk: v for kk, v in eda_features(ep).items()})
        rows.append(r)
R = pd.DataFrame(rows)
R.to_csv(FEAT_DIR / 'recording_features.csv', index=False)
np.savez_compressed(FEAT_DIR / 'signals_30hz.npz', **SIGU)
print(R.shape, '| subjects:', R.subject.nunique(), '| scenario counts:', R.drop_duplicates('subject').scenario.value_counts().to_dict())
print('video duration (s):', R.duration.describe().round(1).to_dict() if 'duration' in R else 'n/a',
      '| fps:', R.fps.median() if 'fps' in R else 'n/a')

  0%|          | 0/56 [00:00<?, ?it/s]

(168, 65) | subjects: 56 | scenario counts: {'ctrl': 30, 'test': 26}
video duration (s): {'count': 168.0, 'mean': 180.0, 'std': 0.0, 'min': 179.9, '25%': 180.0, '50%': 180.0, '75%': 180.0, 'max': 180.0} | fps: 35.138


## 3. Does the physiology respond to the tasks? Is rPPG accurate here?

- **rPPG vs BVP heart rate** per task. Accuracy may drop in T2/T3 because people talk and move.
- **Paired change from rest** (T2 − T1, T3 − T1) per group, with Wilcoxon tests, for HR, HRV, EDA and face motion.

If face motion changes far more than physiology, rPPG features risk acting as talking detectors. Section 4's shortcut control tests exactly that.

In [8]:
from scipy.stats import wilcoxon, spearmanr
if 'rppg_hr' in R and 'bvp_hr' in R:
    for k in (1, 2, 3):
        s = R[R.task == k].dropna(subset=['rppg_hr', 'bvp_hr'])
        e = (s.rppg_hr - s.bvp_hr).abs()
        print(f'T{k}: rPPG vs BVP HR  MAE {e.mean():5.2f} BPM | within 5 BPM {np.mean(e < 5):.0%} | '
              f'Spearman {spearmanr(s.rppg_hr, s.bvp_hr)[0]:+.2f} | n {len(s)}')

KEYS = ['rppg_hr', 'bvp_hr', 'rppg_ln_rmssd', 'bvp_ln_rmssd', 'eda_scl_mean', 'eda_scr_rate', 'mot_step_mean']
KEYS = [k for k in KEYS if k in R]
out = []
for grp in ('test', 'ctrl', 'all'):
    G = R if grp == 'all' else R[R.scenario == grp]
    piv = {k: G.pivot_table(index='subject', columns='task', values=k) for k in KEYS}
    for k in KEYS:
        for t in (2, 3):
            d = (piv[k][t] - piv[k][1]).dropna() if t in piv[k] and 1 in piv[k] else pd.Series(dtype=float)
            if len(d) >= 5:
                out.append(dict(group=grp, feature=k, contrast=f'T{t}-T1', n=len(d), median_change=d.median(),
                                frac_up=(d > 0).mean(), p=wilcoxon(d).pvalue))
resp = pd.DataFrame(out)
print(resp.round(3).to_string(index=False))
resp.to_csv(RES_DIR / 'task_responses.csv', index=False)

T1: rPPG vs BVP HR  MAE  7.45 BPM | within 5 BPM 70% | Spearman +0.54 | n 56
T2: rPPG vs BVP HR  MAE 16.90 BPM | within 5 BPM 36% | Spearman +0.15 | n 56
T3: rPPG vs BVP HR  MAE 17.61 BPM | within 5 BPM 32% | Spearman -0.06 | n 56
group       feature contrast  n  median_change  frac_up     p
 test       rppg_hr    T2-T1 26          1.978    0.577 0.408
 test       rppg_hr    T3-T1 26          1.868    0.577 0.648
 test        bvp_hr    T2-T1 26        -13.125    0.346 0.012
 test        bvp_hr    T3-T1 26         -3.516    0.423 0.069
 test rppg_ln_rmssd    T2-T1 26          0.464    0.885 0.000
 test rppg_ln_rmssd    T3-T1 26          0.266    0.769 0.000
 test  bvp_ln_rmssd    T2-T1 26          0.637    0.846 0.000
 test  bvp_ln_rmssd    T3-T1 26          0.255    0.808 0.001
 test  eda_scl_mean    T2-T1 26          0.335    0.923 0.000
 test  eda_scl_mean    T3-T1 26          0.425    0.808 0.000
 test  eda_scr_rate    T2-T1 26          7.344    0.962 0.000
 test  eda_scr_rate    T3

## 4. Rest vs stress classification, leave-subjects-out

Each feature set is crossed with each setting (**raw**, or **subject-normalised** within each person's three recordings) and each model (logistic regression, RBF-SVM as in the UBFC-Phys paper, random forest).

**How to read the results:**
- A camera-based result is believable only if *rPPG physiology* clearly beats the *quality + motion shortcut*, and only if it agrees with the contact BVP ceiling.
- Rest-identified rate: chance is 33%.

In [9]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score, f1_score

PHYS = ['hr', 'ln_rmssd', 'ln_sdnn', 'pnn50', 'ln_hf', 'lf_hf', 'hr_win_sd']
FEATURE_SETS = {
    'rPPG physiology':                       [f'rppg_{k}' for k in PHYS],
    'rPPG quality + face motion (shortcut)': ['rppg_snr', 'rppg_hr_consistency', 'mot_center_sd', 'mot_step_mean',
                                              'mot_step_p95', 'mot_scale_sd'],
    'contact BVP physiology (ceiling)':      [f'bvp_{k}' for k in PHYS],
    'EDA (wrist)':                           ['eda_scl_mean', 'eda_scl_sd', 'eda_scl_slope', 'eda_scr_rate', 'eda_scr_amp', 'eda_phasic_sd'],
    'BVP + EDA':                             [f'bvp_{k}' for k in PHYS] + ['eda_scl_mean', 'eda_scl_slope', 'eda_scr_rate', 'eda_scr_amp'],
    'rPPG physiology + motion':              [f'rppg_{k}' for k in PHYS] + ['mot_step_mean', 'mot_center_sd'],
}
FEATURE_SETS = {k: [c for c in v if c in R.columns] for k, v in FEATURE_SETS.items()}
CLF = {
    'LogReg':  lambda: LogisticRegression(C=1.0, class_weight='balanced', max_iter=5000),
    'SVM-RBF': lambda: SVC(C=1.0, gamma='scale', class_weight='balanced', probability=True, random_state=0),
    'RF':      lambda: RandomForestClassifier(300, min_samples_leaf=3, class_weight='balanced', n_jobs=-1, random_state=0),
}

def subject_norm(X, groups):
    # z-score each feature within each subject's own recordings (no labels used)
    Z = X.copy()
    for g in np.unique(groups):
        m = groups == g
        mu, sd = X[m].mean(), X[m].std(ddof=0).replace(0, np.nan)
        Z[m] = (X[m] - mu) / sd
    return Z.fillna(0)

def subject_folds(subjects, k, seed):
    rng = np.random.default_rng(seed)
    s = rng.permutation(np.unique(subjects))
    return {sub: i % k for i, sub in enumerate(s)}

def evaluate(D, cols, model, norm, n_rep=N_REPEATS, y_col='stress'):
    D = D.dropna(subset=[y_col]).reset_index(drop=True)
    y, groups = D[y_col].values.astype(int), D.subject.values
    X = D[cols].astype(float)
    if norm:
        X = subject_norm(X, groups)
    prob = np.zeros(len(D))
    for r in range(n_rep):
        fmap = subject_folds(groups, N_FOLDS, seed=100 + r)
        fold = np.array([fmap[g] for g in groups])
        for f in range(N_FOLDS):
            tr, te = fold != f, fold == f
            if te.sum() == 0 or len(np.unique(y[tr])) < 2:
                continue
            pipe = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), CLF[model]())
            prob[te] += pipe.fit(X[tr], y[tr]).predict_proba(X[te])[:, 1] / n_rep
    return D.assign(prob=prob)

def metrics(P_, n_boot=N_BOOT, seed=1):
    y, p = P_.stress.values, P_.prob.values
    def rest_id(Q):
        hits = [g.loc[g.prob.idxmin(), 'task'] == 1 for _, g in Q.groupby('subject') if (g.task == 1).any() and len(g) > 1]
        return np.mean(hits) if hits else np.nan
    out = dict(n_subj=P_.subject.nunique(), auc=roc_auc_score(y, p), acc=accuracy_score(y, p > 0.5),
               bal_acc=balanced_accuracy_score(y, p > 0.5), f1=f1_score(y, p > 0.5), rest_id=rest_id(P_))
    rng, subs = np.random.default_rng(seed), P_.subject.unique()
    bs_auc, bs_acc = [], []
    for _ in range(n_boot):
        pick = rng.choice(subs, len(subs))
        Q = pd.concat([P_[P_.subject == s] for s in pick])
        if Q.stress.nunique() < 2:
            continue
        bs_auc.append(roc_auc_score(Q.stress, Q.prob)); bs_acc.append(accuracy_score(Q.stress, Q.prob > 0.5))
    out.update(auc_lo=np.percentile(bs_auc, 2.5), auc_hi=np.percentile(bs_auc, 97.5),
               acc_lo=np.percentile(bs_acc, 2.5), acc_hi=np.percentile(bs_acc, 97.5))
    return out

res = []
for sname, cols in tqdm(FEATURE_SETS.items()):
    if not cols:
        continue
    for norm in (False, True):
        for model in CLF:
            res.append(dict(set=sname, setting='subject-norm' if norm else 'raw', model=model,
                            **metrics(evaluate(R, cols, model, norm))))
res = pd.DataFrame(res)
pd.set_option('display.width', 200)
print(res.round(3).to_string(index=False))
res.to_csv(RES_DIR / 'rest_vs_stress_all.csv', index=False)

  0%|          | 0/6 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# Same task within each scenario group (test = stressful tasks, ctrl = relaxed tasks); best model per set and setting
best = res.loc[res.groupby(['set', 'setting']).auc.idxmax(), ['set', 'setting', 'model']]
grp_rows = []
for grp in ('test', 'ctrl'):
    G = R[R.scenario == grp]
    if G.subject.nunique() < 8:
        print(f'{grp}: too few subjects ({G.subject.nunique()})'); continue
    for b in best.itertuples():
        grp_rows.append(dict(group=grp, set=b.set, setting=b.setting, model=b.model,
                             **metrics(evaluate(G, FEATURE_SETS[b.set], b.model, b.setting == 'subject-norm'))))
grp_res = pd.DataFrame(grp_rows)
print(grp_res.round(3).to_string(index=False))
grp_res.to_csv(RES_DIR / 'rest_vs_stress_by_group.csv', index=False)

## 5. Transformer on 30 s pulse windows, leave-subjects-out (GPU recommended)

This is the same transformer as week 5 (1 s tokens, 2 layers, 4 heads), set up for rest vs stress. Each window is z-scored, so there's no brightness or skin-tone level. A recording's score is the mean over its windows.
- rPPG pulse.
- Contact BVP pulse (ceiling).

Raw windows only, with no subject normalisation, so this matches the **raw** setting in section 4.

In [10]:
import torch, torch.nn as nn
DEV = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
FS, WIN_SEC = int(FS_RPPG), 30
EPOCHS, PATIENCE, WIN_PER_REC, BATCH = 40, 8, 8, 64
print('device:', DEV)

class PulseTransformer(nn.Module):
    def __init__(self, d=64, heads=4, layers=2, ff=128, drop=0.2):
        super().__init__()
        self.inp = nn.Sequential(nn.Linear(FS, d), nn.LayerNorm(d))
        self.pos = nn.Parameter(torch.randn(1, WIN_SEC, d) * 0.02)
        self.cls = nn.Parameter(torch.zeros(1, 1, d))
        layer = nn.TransformerEncoderLayer(d, heads, ff, drop, batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.head = nn.Sequential(nn.LayerNorm(d), nn.Dropout(drop), nn.Linear(d, 1))
    def forward(self, x):
        x = self.inp(x) + self.pos
        x = self.enc(torch.cat([self.cls.expand(len(x), -1, -1), x], 1))
        return self.head(x[:, 0]).squeeze(-1)

def windows(sig, starts):
    X = np.stack([sig[s * FS:(s + WIN_SEC) * FS] for s in starts]).astype(np.float32)
    X = (X - X.mean(1, keepdims=True)) / (X.std(1, keepdims=True) + 1e-6)
    return torch.from_numpy(X.reshape(len(starts), WIN_SEC, FS)).to(DEV)

def recs(src, D):
    out = []
    for r in D.itertuples():
        s = SIGU.get(f'{src}_s{r.subject}_T{r.task}')
        if s is not None and len(s) >= WIN_SEC * FS:
            out.append((r.subject, r.task, r.stress, s))
    return out

@torch.no_grad()
def score(model, items, stride=10):
    model.eval()
    return np.array([torch.sigmoid(model(windows(s, range(0, len(s) // FS - WIN_SEC + 1, stride)))).mean().item()
                     for _, _, _, s in items])

def train(items, seed):
    rng = np.random.default_rng(seed); torch.manual_seed(seed)
    subs = rng.permutation(sorted({i[0] for i in items}))
    val_s = set(subs[:max(3, len(subs) // 8)])
    fit = [i for i in items if i[0] not in val_s]; val = [i for i in items if i[0] in val_s]
    model = PulseTransformer().to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-2)
    lossf = nn.BCEWithLogitsLoss()
    best, state, bad = np.inf, None, 0
    for ep in range(EPOCHS):
        model.train()
        order = rng.permutation(len(fit) * WIN_PER_REC) % len(fit)
        for b in range(0, len(order), BATCH):
            chunk = [fit[i] for i in order[b:b + BATCH]]
            if len(chunk) < 2:
                continue
            X = torch.cat([windows(s, [rng.integers(0, len(s) // FS - WIN_SEC + 1)]) for _, _, _, s in chunk])
            yb = torch.tensor([c[2] for c in chunk], dtype=torch.float32, device=DEV)
            loss = lossf(model(X), yb)
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        pv = np.clip(score(model, val), 1e-4, 1 - 1e-4)
        yv = np.array([i[2] for i in val])
        vloss = -np.mean(yv * np.log(pv) + (1 - yv) * np.log(1 - pv))
        if vloss < best - 1e-4:
            best, bad, state = vloss, 0, {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(state)
    return model

tf_rows = []
for src, label in (('rppg', 'Transformer: rPPG pulse'), ('bvp', 'Transformer: BVP pulse (ceiling)')):
    items = recs(src, R)
    if len({i[0] for i in items}) < 10:
        print(label, ': too few subjects'); continue
    D = pd.DataFrame([dict(subject=i[0], task=i[1], stress=i[2]) for i in items])
    prob = np.zeros(len(items))
    fmap = subject_folds(D.subject.values, N_FOLDS, seed=100)
    fold = np.array([fmap[s] for s in D.subject])
    for f in range(N_FOLDS):
        tr = [items[i] for i in np.where(fold != f)[0]]
        te_idx = np.where(fold == f)[0]
        prob[te_idx] = score(train(tr, seed=f), [items[i] for i in te_idx])
    tf_rows.append(dict(set=label, setting='raw windows', model='transformer', **metrics(D.assign(prob=prob))))
    print(tf_rows[-1])
tf_res = pd.DataFrame(tf_rows)
tf_res.to_csv(RES_DIR / 'transformer_rest_vs_stress.csv', index=False)

device: cuda
{'set': 'Transformer: rPPG pulse', 'setting': 'raw windows', 'model': 'transformer', 'n_subj': 56, 'auc': np.float64(0.9370216836734695), 'acc': 0.8809523809523809, 'bal_acc': np.float64(0.8526785714285714), 'f1': 0.9130434782608695, 'rest_id': np.float64(0.9642857142857143), 'auc_lo': np.float64(0.8971580038265305), 'auc_hi': np.float64(0.9724210778061225), 'acc_lo': np.float64(0.8331845238095239), 'acc_hi': np.float64(0.9285714285714286)}
{'set': 'Transformer: BVP pulse (ceiling)', 'setting': 'raw windows', 'model': 'transformer', 'n_subj': 56, 'auc': np.float64(0.877391581632653), 'acc': 0.8154761904761905, 'bal_acc': np.float64(0.78125), 'f1': 0.8646288209606987, 'rest_id': np.float64(0.8571428571428571), 'auc_lo': np.float64(0.8274832589285714), 'auc_hi': np.float64(0.9237922512755102), 'acc_lo': np.float64(0.7559523809523809), 'acc_hi': np.float64(0.8690476190476191)}


## 6. Comparison with the literature

Published UBFC-Phys numbers use different splits. The original paper used stratified 7-fold CV, which doesn't say whether it was split by subject. Sensors 2024 used a random sample split. This notebook always holds out whole subjects, which is stricter, so expect lower numbers.

In [11]:
lit = [
    dict(source='UBFC-Phys paper (Meziati Sabour et al., IEEE TAC)', input='remote PRV features, RBF-SVM',
         protocol='stratified 7-fold CV', result='accuracy 85.48%'),
    dict(source='UBFC-Phys paper', input='contact PRV, linear SVM', protocol='stratified 7-fold CV', result='accuracy 75.00%'),
    dict(source='UBFC-Phys paper', input='EDA, RBF-SVM', protocol='stratified 7-fold CV', result='accuracy 82.38%'),
    dict(source='Sensors 2024, 24(4):1096', input='rPPG HR (CHROM), 1-D CNN, T1 vs T2',
         protocol='random 80/10/10 (not subject-wise)', result='accuracy up to 95.83%'),
]
for r in res.sort_values('auc', ascending=False).drop_duplicates('set').itertuples():
    lit.append(dict(source='this work', input=f'{r.set} ({r.setting}, {r.model})', protocol='leave-subjects-out 5-fold x5',
                    result=f'accuracy {100 * r.acc:.1f}% [{100 * r.acc_lo:.1f}, {100 * r.acc_hi:.1f}], AUC {r.auc:.3f}, '
                           f'rest identified {100 * r.rest_id:.0f}%'))
if 'tf_res' in globals() and len(tf_res):
    for r in tf_res.itertuples():
        lit.append(dict(source='this work', input=r.set, protocol='leave-subjects-out 5-fold',
                        result=f'accuracy {100 * r.acc:.1f}%, AUC {r.auc:.3f}, rest identified {100 * r.rest_id:.0f}%'))
lit = pd.DataFrame(lit)
pd.set_option('display.max_colwidth', 90)
print(lit.to_string(index=False))
lit.to_csv(RES_DIR / 'literature_comparison.csv', index=False)

AttributeError: 'list' object has no attribute 'sort_values'

## 7. Summary for the weekly report

In [ ]:
print(f"Subjects: {R.subject.nunique()} (test {R.drop_duplicates('subject').scenario.eq('test').sum()}, "
      f"ctrl {R.drop_duplicates('subject').scenario.eq('ctrl').sum()}) | recordings: {len(R)}")
print('\nRest vs stress, all subjects (best model per set and setting):')
for r in res.sort_values('auc', ascending=False).drop_duplicates(['set', 'setting']).sort_values(['set', 'setting']).itertuples():
    print(f"  {r.set:40s} {r.setting:12s} {r.model:7s} AUC {r.auc:.3f} [{r.auc_lo:.3f}, {r.auc_hi:.3f}] | "
          f"acc {100 * r.acc:.1f}% | bal.acc {100 * r.bal_acc:.1f}% | rest identified {100 * r.rest_id:.0f}% (chance 33%)")
if 'tf_res' in globals() and len(tf_res):
    for r in tf_res.itertuples():
        print(f"  {r.set:40s} {'raw':12s} {'TF':7s} AUC {r.auc:.3f} [{r.auc_lo:.3f}, {r.auc_hi:.3f}] | acc {100 * r.acc:.1f}% "
              f"| rest identified {100 * r.rest_id:.0f}%")
print('\nRead: rPPG works on UBFC-Phys only if "rPPG physiology" clearly beats the "quality + face motion" shortcut,')
print('and its result approaches the contact-BVP ceiling. Overlapping AUC CIs mean no clear difference.')

# Part 2 — Better rPPG, best model, and a talking-free stress test

Part 1 (sections 0–7) showed that rest-vs-task is detected well (AUC ≈ 0.97). But rPPG heart rate was poor during the tasks (17 BPM error vs the wrist sensor), and face motion alone did just as well. Part 2 fixes the signal first, then picks the model fairly:

| Section | What it does |
|---|---|
| 8 | **Better rPPG from the traces already on Drive** (no re-download). 3 methods (POS, CHROM, GREEN) × 4 face regions, with face motion regressed out. Each 15 s window keeps the cleanest candidate, and windows that fail a quality check are dropped. |
| 9 | **Checks the rPPG is good:** heart rate vs the wrist BVP, window by window, before and after the fix. |
| 10 | **Best model by nested cross-validation:** the model and its settings are chosen *inside* the training subjects only, so the reported AUC/accuracy is not inflated by picking the best of many runs. Includes rPPG with motion statistically removed, and a paired test of rPPG vs the motion shortcut. |
| 11 | **Stress without the talking confound:** stressful (test) vs relaxed (ctrl) version of the *same* task. Both groups talk, so motion can't explain a difference. |
| 12 | Saves the best models to Drive, and prints the summary. |

**After a restart**, run cell 3 (setup), cell 11 (signal functions) and then this part. It reloads everything it needs from Drive. Runtime: section 8 about 10–20 min, section 10 about 15–30 min, the rest a few minutes. No GPU needed.

## 8. Better rPPG extraction (from the saved traces)

Why the old signal failed during tasks: speaking moves the jaw and head, and the face box (which includes the mouth) picks up those movements as colour changes at heart-rate-like frequencies. Improvements:
1. **Motion regression.** The face-box track (x, y, size and their velocities) is band-passed to the heart-rate band and regressed out of each candidate signal in 10 s blocks.
2. **Several candidates:** POS, CHROM and GREEN on each of 4 regions (whole face, forehead, left cheek, right cheek) gives 12 signals. The forehead moves least when talking.
3. **Window selection:** in each 15 s window (5 s step), the candidate with the highest SNR is kept.
4. **Quality gate:** a window is used only if its SNR ≥ `SNR_MIN` and its heart rate is within `HR_JUMP` BPM of the recording's median good-window rate. Otherwise it's dropped, and the **coverage** (fraction of windows kept) is recorded.
5. **HRV only from beats inside good windows.**

Thresholds are fixed in advance and do not use any stress labels.

In [12]:
# 8.0 Helpers (self-contained, so this part also runs after a restart)
from numpy.lib.stride_tricks import sliding_window_view
from scipy.signal import resample_poly
from scipy.stats import theilslopes, spearmanr, mannwhitneyu

SNR_MIN, HR_JUMP = 0.0, 15.0        # quality gate (dB, BPM); see the sweep in section 9
WIN_S, STEP_S, BLOCK_S = 15, 5, 10  # analysis window, step, motion-regression block (s)
ROI_ALL = ['face', 'forehead', 'left_cheek', 'right_cheek']

def load_col(path):
    try:
        x = pd.to_numeric(pd.read_csv(path, header=None).iloc[:, 0], errors='coerce').dropna().values
    except Exception:
        x = np.array([])
    return x.astype(float)

def _overlap_add(h, N, l):
    H = np.zeros(N)
    for j in range(l):
        H[j:j + len(h)] += h[:, j]
    return H

def pos_fast(rgb, fs, win=1.6):
    N, l = len(rgb), int(round(win * fs))
    Wv = sliding_window_view(rgb, l, axis=0)
    mu = Wv.mean(axis=2, keepdims=True); mu[mu == 0] = 1e-9
    Cn = Wv / mu
    S0, S1 = Cn[:, 1] - Cn[:, 2], -2 * Cn[:, 0] + Cn[:, 1] + Cn[:, 2]
    sd1 = S1.std(axis=1, keepdims=True)
    h = S0 + (S0.std(axis=1, keepdims=True) / np.where(sd1 > 0, sd1, np.inf)) * S1
    return _overlap_add(h - h.mean(axis=1, keepdims=True), N, l)

def chrom_fast(rgb, fs, win=1.6):
    # CHROM (de Haan & Jeanne 2013), sliding-window form like pos_fast.
    N, l = len(rgb), int(round(win * fs))
    Wv = sliding_window_view(rgb, l, axis=0)
    mu = Wv.mean(axis=2, keepdims=True); mu[mu == 0] = 1e-9
    Cn = Wv / mu
    X = 3 * Cn[:, 0] - 2 * Cn[:, 1]
    Y = 1.5 * Cn[:, 0] + Cn[:, 1] - 1.5 * Cn[:, 2]
    sy = Y.std(axis=1, keepdims=True)
    h = X - (X.std(axis=1, keepdims=True) / np.where(sy > 0, sy, np.inf)) * Y
    return _overlap_add(h - h.mean(axis=1, keepdims=True), N, l)

def green(rgb, fs, win=1.6):
    g = rgb[:, 1]
    l = int(round(win * fs))
    m = pd.Series(g).rolling(l, center=True, min_periods=1).mean().values
    return g / np.where(m == 0, 1e-9, m) - 1

METHODS = {'POS': pos_fast, 'CHROM': chrom_fast, 'GREEN': green}

def motion_regressors(boxes, t, tu):
    bb = pd.DataFrame(boxes).interpolate(limit_direction='both').values
    if not np.isfinite(bb).all():
        return None
    cx, cy, w = bb[:, 0] + bb[:, 2] / 2, bb[:, 1] + bb[:, 3] / 2, bb[:, 2]
    M = []
    for v in (cx, cy, w):
        vu = np.interp(tu, t, v)
        vb = bandpass(vu - vu.mean(), FS_RPPG, *HR_BAND)
        M += [vb, np.gradient(vb)]
    return np.column_stack(M)

def regress_out(s, M, block):
    # least-squares removal of motion components, block by block
    if M is None:
        return s
    out = s.copy()
    for a in range(0, len(s), block):
        b = min(len(s), a + block)
        if b - a < 30:
            continue
        A = np.column_stack([M[a:b], np.ones(b - a)])
        coef, *_ = np.linalg.lstsq(A, s[a:b], rcond=None)
        out[a:b] = s[a:b] - A @ coef
    return out

def window_hr(x, fs, n, step):
    starts = np.arange(0, len(x) - n + 1, step)
    hr, snr = np.empty(len(starts)), np.empty(len(starts))
    for i, a in enumerate(starts):
        h, f, P = hr_harmonic(x[a:a + n], fs)
        hr[i], snr[i] = h, snr_db(f, P, h / 60)
    return starts, hr, snr

def hrv_masked(pk, fs, keep_mask):
    # HRV from beats whose two ends both fall inside good samples
    if len(pk) < 10:
        return {}
    t = pk / fs
    ibi = np.diff(t)
    ok = keep_mask[pk[:-1]] & keep_mask[pk[1:]] & (ibi > 0.33) & (ibi < 1.5)
    med = pd.Series(np.where(ok, ibi, np.nan)).rolling(7, center=True, min_periods=1).median().values
    ok &= np.abs(ibi - med) < 0.2 * med
    if ok.sum() < 10:
        return dict(n_ibi=int(ok.sum()))
    d = np.diff(ibi)[ok[1:] & ok[:-1]]
    sd = 1000 * ibi[ok].std(ddof=1)
    rm = 1000 * np.sqrt(np.mean(d ** 2)) if len(d) else np.nan
    return dict(n_ibi=int(ok.sum()), sdnn=sd, rmssd=rm, ln_sdnn=np.log(max(sd, 1e-3)),
                ln_rmssd=np.log(max(rm, 1e-3)) if np.isfinite(rm) else np.nan,
                pnn50=np.mean(np.abs(d) > 0.05) if len(d) else np.nan)
print('helpers ready')

helpers ready


In [13]:
# 8.1 Improved rPPG for every recording, plus window-level BVP reference
def rppg_v2(trace_path, bvp_path=None):
    z = np.load(trace_path)
    rgb = z['traces'].astype(float); dup = z['dup'].astype(bool); rgb[dup] = np.nan
    fps = float(z['fps_container']); fps = fps if 20 < fps < 61 else DEFAULT_FPS
    t = np.arange(len(rgb)) / fps
    tu = np.arange(0, t[-1], 1 / FS_RPPG)
    M = motion_regressors(z['boxes'], t, tu) if 'boxes' in z.files else None
    names = [str(n) for n in z['roi_names']] if 'roi_names' in z.files else ROI_ALL
    n, step, block = int(WIN_S * FS_RPPG), int(STEP_S * FS_RPPG), int(BLOCK_S * FS_RPPG)
    cands = {}
    for k, roi in enumerate(names):
        c = interp_nans(rgb[:, k, :])
        if c is None:
            continue
        cu = np.column_stack([np.interp(tu, t, c[:, j]) for j in range(3)])
        for mname, fn in METHODS.items():
            s = bandpass(fn(cu, FS_RPPG), FS_RPPG, *HR_BAND)
            s = regress_out(s, M, block)
            cands[f'{mname}-{roi}'] = s / (s.std() + 1e-12)
    if not cands:
        return {}, None
    keys = list(cands)
    W = {k: window_hr(cands[k], FS_RPPG, n, step) for k in keys}
    starts = W[keys[0]][0]
    hr_all = np.array([W[k][1] for k in keys]); snr_all = np.array([W[k][2] for k in keys])
    best = np.nanargmax(np.nan_to_num(snr_all, nan=-99), axis=0)
    hr_w = hr_all[best, np.arange(len(starts))]; snr_w = snr_all[best, np.arange(len(starts))]
    good = snr_w >= SNR_MIN
    if good.sum() >= 3:                              # continuity check around the median of good windows
        good &= np.abs(hr_w - np.median(hr_w[good])) <= HR_JUMP
    win = pd.DataFrame(dict(start_s=starts / FS_RPPG, hr=hr_w, snr=snr_w, good=good,
                            cand=[keys[i] for i in best]))
    # BVP in the same windows (reference only)
    if bvp_path is not None:
        b = load_col(bvp_path)
        if len(b) > 30 * BVP_FS:
            b = bandpass(detrend(b), BVP_FS, *HR_BAND)
            nb_, sb_ = int(WIN_S * BVP_FS), int(STEP_S * BVP_FS)
            bs, bhr, bsnr = window_hr(b, BVP_FS, nb_, sb_)
            m = min(len(win), len(bhr))
            win.loc[:m - 1, 'bvp_hr'] = bhr[:m]; win.loc[:m - 1, 'bvp_snr'] = bsnr[:m]
    o = dict(coverage=good.mean(), snr_mean=np.nanmean(snr_w), snr_good=np.nanmean(snr_w[good]) if good.any() else np.nan,
             forehead_share=np.mean(['forehead' in c for c in win.cand]))
    if good.sum() >= 2:
        hg, tg = hr_w[good], starts[good] / FS_RPPG
        o.update(hr=np.median(hg), hr_iqr=np.subtract(*np.percentile(hg, [75, 25])),
                 hr_slope=theilslopes(hg, tg / 60)[0] if good.sum() >= 3 else np.nan)
        # HRV: the candidate that wins most good windows, beats inside good windows only
        main = keys[np.bincount(best[good], minlength=len(keys)).argmax()]
        keep = np.zeros(len(cands[main]), bool)
        for a in starts[good]:
            keep[a:a + n] = True
        pk = detect_beats(cands[main], FS_RPPG, o['hr'])
        o.update(hrv_masked(pk, FS_RPPG, keep))
        o['main_cand'] = main
    return o, win

rows2, WIN = [], []
recs = R[['subject', 'task']].drop_duplicates().itertuples(index=False) if 'R' in globals() else \
       [(n, k) for n in range(1, 57) for k in (1, 2, 3) if (TRACES / f'vid_s{n}_T{k}.npz').exists()]
for n, k in tqdm(list(recs)):
    tp = TRACES / f'vid_s{n}_T{k}.npz'
    bp = next(iter((SMALL / f's{n}').glob(f'bvp_s{n}_T{k}.csv')), None)
    try:
        o, w = rppg_v2(tp, bp)
    except Exception as e:
        o, w = dict(error=repr(e)[:120]), None
    rows2.append(dict(subject=n, task=k, **{'rp2_' + kk: v for kk, v in o.items()}))
    if w is not None:
        WIN.append(w.assign(subject=n, task=k))
RP2 = pd.DataFrame(rows2)
WINS = pd.concat(WIN, ignore_index=True)
RP2.to_csv(FEAT_DIR / 'rppg_v2_features.csv', index=False)
WINS.to_csv(FEAT_DIR / 'rppg_v2_windows.csv', index=False)
print('recordings:', len(RP2), '| windows:', len(WINS), '| errors:', RP2.get('rp2_error', pd.Series(dtype=str)).notna().sum())
print('winning candidates (share of windows):')
print(WINS.cand.str.split('-').str[0].value_counts(normalize=True).round(2).to_dict(),
      WINS.cand.str.split('-', n=1).str[1].value_counts(normalize=True).round(2).to_dict())

  0%|          | 0/168 [00:00<?, ?it/s]

recordings: 168 | windows: 5707 | errors: 0
winning candidates (share of windows):
{'POS': 0.52, 'CHROM': 0.26, 'GREEN': 0.22} {'face': 0.38, 'forehead': 0.36, 'right_cheek': 0.13, 'left_cheek': 0.13}


In [14]:
# 8.2 Merge with the part-1 features (reloads them from Drive after a restart)
if 'R' not in globals():
    R = pd.read_csv(FEAT_DIR / 'recording_features.csv')
R2 = R.drop(columns=[c for c in R.columns if c.startswith('rp2_')]).merge(RP2, on=['subject', 'task'], how='left')
R2.to_csv(FEAT_DIR / 'recording_features_v2.csv', index=False)
print(R2.shape, '| subjects:', R2.subject.nunique())

(168, 79) | subjects: 56


## 9. Is the rPPG good now? Validation against the wrist BVP

- **Recording level:** heart-rate error of the old (part 1) and new rPPG vs the BVP, per task.
- **Window level:** error in kept vs dropped windows, which shows whether the quality gate works.
- **Gate sweep:** error vs coverage for several SNR thresholds.

**Caveat:** the wrist BVP is itself disturbed by gesturing during the tasks. The cleanest comparison uses only windows where the BVP is also clean (BVP SNR ≥ 3 dB).

In [15]:
def mae_tbl(df, a, b):
    d = df[[a, b]].dropna()
    e = (d[a] - d[b]).abs()
    return dict(n=len(d), mae=e.mean(), median_ae=e.median(), within5=(e < 5).mean(),
                spearman=spearmanr(d[a], d[b])[0] if len(d) > 2 else np.nan)

rows = []
for k in (1, 2, 3):
    D = R2[R2.task == k]
    rows.append(dict(task=f'T{k}', signal='old rPPG (part 1)', **mae_tbl(D, 'rppg_hr', 'bvp_hr')))
    rows.append(dict(task=f'T{k}', signal='new rPPG v2 (gated)', coverage=D.rp2_coverage.mean(),
                     **mae_tbl(D, 'rp2_hr', 'bvp_hr')))
rec_val = pd.DataFrame(rows)
print('Recording level: rPPG vs wrist BVP heart rate')
print(rec_val.round(3).to_string(index=False))

Wc = WINS.dropna(subset=['bvp_hr'])
Wc = Wc.assign(err=(Wc.hr - Wc.bvp_hr).abs(), bvp_clean=Wc.bvp_snr >= 3)
print('\nWindow level (only windows where the BVP itself is clean):')
print(Wc[Wc.bvp_clean].groupby(['task', 'good']).err.agg(['count', 'mean', 'median',
      lambda e: (e < 5).mean()]).rename(columns={'<lambda_0>': 'within5'}).round(2).to_string())

print('\nGate sweep (BVP-clean windows): SNR threshold -> coverage and error of kept windows')
sw = []
for th in (-3, -1, 0, 1, 2, 3, 5):
    for k in (1, 2, 3):
        X = Wc[(Wc.task == k) & Wc.bvp_clean]
        kept = X[X.snr >= th]
        sw.append(dict(snr_min=th, task=f'T{k}', coverage=len(kept) / max(len(X), 1),
                       mae=kept.err.mean(), within5=(kept.err < 5).mean()))
print(pd.DataFrame(sw).pivot(index='snr_min', columns='task', values=['coverage', 'mae']).round(2).to_string())
rec_val.to_csv(RES_DIR / 'rppg_v2_validation.csv', index=False)

Recording level: rPPG vs wrist BVP heart rate
task              signal  n    mae  median_ae  within5  spearman  coverage
  T1   old rPPG (part 1) 56  7.446      0.549    0.696     0.545       NaN
  T1 new rPPG v2 (gated) 56  5.853      0.648    0.732     0.706     0.855
  T2   old rPPG (part 1) 56 16.904      8.840    0.357     0.154       NaN
  T2 new rPPG v2 (gated) 56 17.959     11.316    0.321     0.159     0.470
  T3   old rPPG (part 1) 56 17.608     10.774    0.321    -0.063       NaN
  T3 new rPPG v2 (gated) 56 17.611      9.170    0.375    -0.019     0.556

Window level (only windows where the BVP itself is clean):
            count   mean  median  within5
task good                                
1    False    112  23.44   21.64     0.18
     True    1094   3.42    0.86     0.83
2    False    141  27.90   27.19     0.14
     True     171   8.76    1.76     0.73
3    False    134  27.34   28.56     0.19
     True     298   5.03    2.67     0.73

Gate sweep (BVP-clean windows): 

## 10. Best model: nested leave-subjects-out selection

For each feature set:
- **Outer loop:** 5 subject folds × 3 repeats, used for reporting.
- **Inner loop:** 4 subject folds inside the training subjects, used to choose among logistic regression, RBF-SVM, random forest, extra trees and gradient boosting, each with a small hyper-parameter grid, by AUC.

The reported numbers come only from the outer test subjects, so picking the best model can't inflate them.

**Feature sets:**
- rPPG v1 (part 1) and **rPPG v2** physiology: heart rate, its spread and slope, and gated HRV.
- **rPPG v2, motion removed:** inside each training fold, face motion is regressed out of every rPPG feature. This is what rPPG adds beyond "the person is moving".
- Motion + signal quality (shortcut).
- rPPG v2 + motion (best camera-only).
- Contact references: BVP, EDA, BVP + EDA.

**Settings:**
- *subject-norm* compares each recording with the same person's other recordings. This needs a personal baseline at deployment.
- *raw* uses each recording alone.

The last table is a **paired test**: AUC of rPPG v2 minus AUC of the motion shortcut, on the same bootstrap subjects.

In [16]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score, f1_score

N_REP_NESTED, N_INNER, N_BOOT2 = 3, 4, 1000

class Residualize(BaseEstimator, TransformerMixin):
    # Regress the first n_phys columns on the remaining (motion) columns using training rows only;
    # output the physiology residuals (motion columns are dropped).
    def __init__(self, n_phys=0):
        self.n_phys = n_phys
    def fit(self, X, y=None):
        X = np.asarray(X, float)
        self.lr_ = LinearRegression().fit(X[:, self.n_phys:], X[:, :self.n_phys]) if self.n_phys else None
        return self
    def transform(self, X):
        X = np.asarray(X, float)
        return X[:, :self.n_phys] - self.lr_.predict(X[:, self.n_phys:]) if self.n_phys else X

GRID = [
    {'clf': [LogisticRegression(class_weight='balanced', max_iter=5000)], 'clf__C': [0.1, 1, 10]},
    {'clf': [SVC(class_weight='balanced', probability=True, random_state=0)], 'clf__C': [0.3, 1, 3, 10], 'clf__gamma': ['scale']},
    {'clf': [RandomForestClassifier(200, class_weight='balanced', random_state=0, n_jobs=1)], 'clf__min_samples_leaf': [1, 3, 8]},
    {'clf': [ExtraTreesClassifier(200, class_weight='balanced', random_state=0, n_jobs=1)], 'clf__min_samples_leaf': [1, 3]},
    {'clf': [HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=150, random_state=0)],
     'clf__min_samples_leaf': [5, 15]},
]

def make_search(n_phys_resid=0):
    steps = [('imp', SimpleImputer(strategy='median'))]
    if n_phys_resid:
        steps.append(('res', Residualize(n_phys_resid)))
    steps += [('sc', StandardScaler()), ('clf', LogisticRegression())]
    return GridSearchCV(Pipeline(steps), GRID, scoring='roc_auc', cv=GroupKFold(N_INNER), n_jobs=-1, refit=True)

def subject_norm(X, groups):
    Z = X.copy()
    for g in np.unique(groups):
        m = groups == g
        mu, sd = X[m].mean(), X[m].std(ddof=0).replace(0, np.nan)
        Z[m] = (X[m] - mu) / sd
    return Z.fillna(0)

def subject_folds(subjects, k, seed):
    s = np.random.default_rng(seed).permutation(np.unique(subjects))
    return {sub: i % k for i, sub in enumerate(s)}

def nested_eval(D, cols, norm, resid_motion=None, y_col='stress', n_rep=N_REP_NESTED):
    D = D.dropna(subset=[y_col]).reset_index(drop=True)
    use = list(cols) + (list(resid_motion) if resid_motion else [])
    y, g = D[y_col].values.astype(int), D.subject.values
    X = D[use].astype(float)
    if norm:
        X = subject_norm(X, g)
    prob, chosen = np.zeros(len(D)), []
    for r in range(n_rep):
        fmap = subject_folds(g, N_FOLDS, seed=100 + r)
        fold = np.array([fmap[s] for s in g])
        for f in range(N_FOLDS):
            tr, te = fold != f, fold == f
            if te.sum() == 0 or len(np.unique(y[tr])) < 2:
                continue
            gs = make_search(len(cols) if resid_motion else 0).fit(X[tr], y[tr], groups=g[tr])
            prob[te] += gs.predict_proba(X[te])[:, 1] / n_rep
            chosen.append(type(gs.best_params_['clf']).__name__.replace('Classifier', ''))
    return D.assign(prob=prob, label=y), pd.Series(chosen).value_counts().to_dict()

def boot_idx(subjects, n_boot, seed):
    rng, subs = np.random.default_rng(seed), np.unique(subjects)
    return [rng.choice(subs, len(subs)) for _ in range(n_boot)]

def metrics2(P_, boots):
    y, p = P_.label.values, P_.prob.values
    hits = [g.loc[g.prob.idxmin(), 'task'] == 1 for _, g in P_.groupby('subject')
            if 'task' in g and (g.task == 1).any() and len(g) > 1]
    out = dict(n_subj=P_.subject.nunique(), auc=roc_auc_score(y, p), acc=accuracy_score(y, p > 0.5),
               bal_acc=balanced_accuracy_score(y, p > 0.5), f1=f1_score(y, p > 0.5),
               rest_id=np.mean(hits) if hits else np.nan)
    by = {s: P_[P_.subject == s] for s in P_.subject.unique()}
    a, c = [], []
    for pick in boots:
        Q = pd.concat([by[s] for s in pick])
        if Q.label.nunique() < 2:
            continue
        a.append(roc_auc_score(Q.label, Q.prob)); c.append(accuracy_score(Q.label, Q.prob > 0.5))
    out.update(auc_lo=np.percentile(a, 2.5), auc_hi=np.percentile(a, 97.5),
               acc_lo=np.percentile(c, 2.5), acc_hi=np.percentile(c, 97.5))
    return out

def paired_delta(PA, PB, boots):
    # AUC(A) - AUC(B) on identical bootstrap subject samples
    byA = {s: PA[PA.subject == s] for s in PA.subject.unique()}
    byB = {s: PB[PB.subject == s] for s in PB.subject.unique()}
    d = []
    for pick in boots:
        A = pd.concat([byA[s] for s in pick]); B = pd.concat([byB[s] for s in pick])
        if A.label.nunique() < 2:
            continue
        d.append(roc_auc_score(A.label, A.prob) - roc_auc_score(B.label, B.prob))
    return np.mean(d), np.percentile(d, 2.5), np.percentile(d, 97.5)

PHYS1 = ['rppg_hr', 'rppg_ln_rmssd', 'rppg_ln_sdnn', 'rppg_pnn50', 'rppg_ln_hf', 'rppg_lf_hf', 'rppg_hr_win_sd']
PHYS2 = ['rp2_hr', 'rp2_hr_iqr', 'rp2_hr_slope', 'rp2_ln_rmssd', 'rp2_ln_sdnn', 'rp2_pnn50']
MOTION = ['mot_center_sd', 'mot_step_mean', 'mot_step_p95', 'mot_scale_sd']
QUALITY = ['rp2_coverage', 'rp2_snr_mean', 'rppg_snr', 'rppg_hr_consistency']
BVP = ['bvp_hr', 'bvp_ln_rmssd', 'bvp_ln_sdnn', 'bvp_pnn50', 'bvp_ln_hf', 'bvp_lf_hf', 'bvp_hr_win_sd']
EDA = ['eda_scl_mean', 'eda_scl_sd', 'eda_scl_slope', 'eda_scr_rate', 'eda_scr_amp', 'eda_phasic_sd']
keep = lambda L: [c for c in L if c in R2.columns]
SETS10 = {   # name: (feature columns, motion columns to regress out or None)
    'rPPG v1 physiology (part 1)':      (keep(PHYS1), None),
    'rPPG v2 physiology':               (keep(PHYS2), None),
    'rPPG v2 physiology, motion removed': (keep(PHYS2), keep(MOTION)),
    'motion + quality (shortcut)':      (keep(MOTION + QUALITY), None),
    'rPPG v2 + motion (camera only)':   (keep(PHYS2 + MOTION), None),
    'contact BVP (ceiling)':            (keep(BVP), None),
    'EDA (wrist)':                      (keep(EDA), None),
    'BVP + EDA':                        (keep(BVP + EDA), None),
}
boots = boot_idx(R2.subject.values, N_BOOT2, seed=1)
res10, PRED10 = [], {}
for name, (cols, resid) in tqdm(SETS10.items()):
    for norm in (True, False):
        setting = 'subject-norm' if norm else 'raw'
        Pn, chosen = nested_eval(R2, cols, norm, resid)
        PRED10[(name, setting)] = Pn
        res10.append(dict(set=name, setting=setting, n_feat=len(cols), **metrics2(Pn, boots),
                          models_chosen=chosen))
res10 = pd.DataFrame(res10)
pd.set_option('display.width', 250); pd.set_option('display.max_colwidth', 70)
print(res10.drop(columns='models_chosen').round(3).to_string(index=False))
print('\nModels picked by the inner loop:')
for r in res10.itertuples():
    print(f'  {r.set:38s} {r.setting:12s} {r.models_chosen}')
res10.to_csv(RES_DIR / 'nested_rest_vs_task.csv', index=False)

  0%|          | 0/8 [00:00<?, ?it/s]

                               set      setting  n_feat  n_subj   auc   acc  bal_acc    f1  rest_id  auc_lo  auc_hi  acc_lo  acc_hi
       rPPG v1 physiology (part 1) subject-norm       7      56 0.962 0.881    0.862 0.912    0.839   0.926   0.987   0.821   0.940
       rPPG v1 physiology (part 1)          raw       7      56 0.898 0.863    0.830 0.900    0.857   0.850   0.947   0.815   0.911
                rPPG v2 physiology subject-norm       6      56 0.823 0.786    0.759 0.839    0.643   0.732   0.897   0.714   0.851
                rPPG v2 physiology          raw       6      56 0.817 0.786    0.741 0.845    0.786   0.761   0.872   0.738   0.833
rPPG v2 physiology, motion removed subject-norm       6      56 0.785 0.750    0.679 0.826    0.696   0.706   0.857   0.690   0.810
rPPG v2 physiology, motion removed          raw       6      56 0.733 0.714    0.616 0.810    0.607   0.654   0.804   0.673   0.762
       motion + quality (shortcut) subject-norm       8      56 0.995 0.976 

In [17]:
# Paired comparisons: does rPPG add anything beyond motion, and did v2 improve on v1?
pairs = [('rPPG v2 physiology', 'motion + quality (shortcut)'),
         ('rPPG v2 physiology, motion removed', 'motion + quality (shortcut)'),
         ('rPPG v2 + motion (camera only)', 'motion + quality (shortcut)'),
         ('rPPG v2 physiology', 'rPPG v1 physiology (part 1)'),
         ('rPPG v2 physiology', 'contact BVP (ceiling)')]
pr = []
for a, b in pairs:
    for setting in ('subject-norm', 'raw'):
        m, lo, hi = paired_delta(PRED10[(a, setting)], PRED10[(b, setting)], boots)
        pr.append(dict(A=a, B=b, setting=setting, dAUC=m, lo=lo, hi=hi,
                       verdict='A better' if lo > 0 else ('B better' if hi < 0 else 'no clear difference')))
pr = pd.DataFrame(pr)
print(pr.round(3).to_string(index=False))
pr.to_csv(RES_DIR / 'nested_paired_comparisons.csv', index=False)

                                 A                           B      setting   dAUC     lo     hi             verdict
                rPPG v2 physiology motion + quality (shortcut) subject-norm -0.174 -0.263 -0.099            B better
                rPPG v2 physiology motion + quality (shortcut)          raw -0.143 -0.204 -0.087            B better
rPPG v2 physiology, motion removed motion + quality (shortcut) subject-norm -0.209 -0.287 -0.138            B better
rPPG v2 physiology, motion removed motion + quality (shortcut)          raw -0.228 -0.309 -0.150            B better
    rPPG v2 + motion (camera only) motion + quality (shortcut) subject-norm -0.027 -0.060 -0.004            B better
    rPPG v2 + motion (camera only) motion + quality (shortcut)          raw  0.011 -0.003  0.030 no clear difference
                rPPG v2 physiology rPPG v1 physiology (part 1) subject-norm -0.140 -0.231 -0.063            B better
                rPPG v2 physiology rPPG v1 physiology (part 1)  

## 11. Stress without the talking confound: stressful vs relaxed version of the same task

Half the participants did the **test** (stressful) version of the speech and arithmetic tasks, the other half the **ctrl** (relaxed) version. Both versions involve talking, so comparing test vs ctrl within T2 (and within T3) asks whether rPPG detects **stress**, not speech.

- Features are each person's **change from their own rest** (task − T1), which removes between-person differences.
- Label: test = 1, ctrl = 0. 56 people per task; T2 and T3 rows are pooled, grouped by subject.
- Same nested model selection as section 10.
- Manipulation check: did the test group report more anxiety?

This is much harder than rest vs task (between-person, 56 people), so expect far lower numbers. An AUC whose CI stays above 0.5 here is genuine evidence of stress detection.

In [18]:
# manipulation check on self-reported anxiety (columns anx_*, as read from selfReportedAnx_sN.csv)
info = R2.drop_duplicates('subject').set_index('subject')
anx_cols = [c for c in info.columns if c.startswith('anx_')]
if anx_cols:
    print('Self-reported anxiety, test vs ctrl (Mann-Whitney):')
    for c in anx_cols:
        a, b = info.loc[info.scenario == 'test', c].dropna(), info.loc[info.scenario == 'ctrl', c].dropna()
        if len(a) > 3 and len(b) > 3:
            print(f'  {c}: test median {a.median():.1f} vs ctrl {b.median():.1f} | p = {mannwhitneyu(a, b).pvalue:.3f}')

delta_cols = keep(PHYS2 + PHYS1 + MOTION + QUALITY + BVP + EDA)
base = R2[R2.task == 1].set_index('subject')[delta_cols]
rows = []
for k in (2, 3):
    T = R2[R2.task == k].set_index('subject')
    Dk = (T[delta_cols] - base.reindex(T.index)).add_prefix('d_')
    Dk['subject'], Dk['task'] = Dk.index, k
    Dk['label_test'] = (T.scenario == 'test').astype(int)
    rows.append(Dk.reset_index(drop=True))
DT = pd.concat(rows, ignore_index=True).dropna(subset=['label_test'])
print('\nrows:', len(DT), '| subjects:', DT.subject.nunique(), '| test share: %.2f' % DT.label_test.mean())

d = lambda L: ['d_' + c for c in keep(L)]
SETS11 = {
    'rPPG v2 change':                    (d(PHYS2), None),
    'rPPG v2 change, motion removed':    (d(PHYS2), d(MOTION)),
    'motion + quality change (shortcut)': (d(MOTION + QUALITY), None),
    'rPPG v2 + motion change':           (d(PHYS2 + MOTION), None),
    'contact BVP change (ceiling)':      (d(BVP), None),
    'EDA change':                        (d(EDA), None),
    'BVP + EDA change':                  (d(BVP + EDA), None),
}
boots11 = boot_idx(DT.subject.values, N_BOOT2, seed=2)
res11, PRED11 = [], {}
for name, (cols, resid) in tqdm(SETS11.items()):
    Pn, chosen = nested_eval(DT, cols, norm=False, resid_motion=resid, y_col='label_test')
    PRED11[name] = Pn
    res11.append(dict(set=name, n_feat=len(cols), **{k: v for k, v in metrics2(Pn, boots11).items() if k != 'rest_id'},
                      models_chosen=chosen))
res11 = pd.DataFrame(res11)
print(res11.drop(columns='models_chosen').round(3).to_string(index=False))
res11.to_csv(RES_DIR / 'nested_test_vs_ctrl.csv', index=False)
print('\nChance: AUC 0.5 | accuracy of always guessing "ctrl" = %.2f' % (1 - DT.label_test.mean()))

Self-reported anxiety, test vs ctrl (Mann-Whitney):
  anx_1: test median 1.9 vs ctrl 1.6 | p = 0.318
  anx_2: test median 2.1 vs ctrl 1.6 | p = 0.071
  anx_3: test median 1.6 vs ctrl 1.7 | p = 0.974
  anx_4: test median 1.9 vs ctrl 1.7 | p = 0.091
  anx_5: test median 2.4 vs ctrl 2.7 | p = 0.343
  anx_6: test median 2.2 vs ctrl 2.6 | p = 0.157

rows: 112 | subjects: 56 | test share: 0.46


  0%|          | 0/7 [00:00<?, ?it/s]

                               set  n_feat  n_subj   auc   acc  bal_acc    f1  auc_lo  auc_hi  acc_lo  acc_hi
                    rPPG v2 change       6      56 0.401 0.411    0.401 0.298   0.283   0.524   0.312   0.518
    rPPG v2 change, motion removed       6      56 0.540 0.482    0.481 0.453   0.433   0.646   0.384   0.571
motion + quality change (shortcut)       8      56 0.518 0.545    0.540 0.495   0.410   0.626   0.446   0.643
           rPPG v2 + motion change      10      56 0.527 0.527    0.522 0.475   0.415   0.643   0.438   0.616
      contact BVP change (ceiling)       7      56 0.405 0.491    0.484 0.412   0.266   0.543   0.375   0.598
                        EDA change       6      56 0.490 0.500    0.495 0.440   0.362   0.616   0.393   0.598
                  BVP + EDA change      13      56 0.409 0.473    0.469 0.416   0.282   0.540   0.366   0.580

Chance: AUC 0.5 | accuracy of always guessing "ctrl" = 0.54


## 12. Save the best models and summarise

Two camera-only models are refit on **all 56 subjects** and saved to `results/best_models.joblib`, with their feature lists and settings. Each uses the model type and settings that the inner loop picked most often, re-selected on all data.
- **Best rest-vs-task model** (highest nested AUC among camera-only sets).
- **Best rPPG-physiology-only model** (no motion features).

The headline numbers to report are the **nested outer-fold** results from sections 10–11, not the fit on all data.

In [19]:
import joblib
cam_sets = ['rPPG v2 physiology', 'rPPG v2 physiology, motion removed', 'rPPG v2 + motion (camera only)',
            'motion + quality (shortcut)', 'rPPG v1 physiology (part 1)']
cam = res10[res10.set.isin(cam_sets)].sort_values('auc', ascending=False)
best_any = cam.iloc[0]
best_phys = cam[cam.set.str.startswith('rPPG v2 physiology')].iloc[0]
saved = {}
for tag, row in (('best_camera', best_any), ('best_rppg_physiology', best_phys)):
    cols, resid = SETS10[row.set]
    use = list(cols) + (list(resid) if resid else [])
    D = R2.dropna(subset=['stress']).reset_index(drop=True)
    X = D[use].astype(float)
    if row.setting == 'subject-norm':
        X = subject_norm(X, D.subject.values)
    gs = make_search(len(cols) if resid else 0).fit(X, D.stress.values, groups=D.subject.values)
    saved[tag] = dict(model=gs.best_estimator_, features=use, setting=row.setting, set=row.set,
                      nested_auc=row.auc, nested_acc=row.acc, params=str(gs.best_params_))
    print(f"{tag}: {row.set} [{row.setting}] -> {gs.best_params_['clf'].__class__.__name__} | "
          f"nested AUC {row.auc:.3f} [{row.auc_lo:.3f}, {row.auc_hi:.3f}], accuracy {100 * row.acc:.1f}%")
joblib.dump(saved, RES_DIR / 'best_models.joblib')
print('saved ->', RES_DIR / 'best_models.joblib')

best_camera: motion + quality (shortcut) [subject-norm] -> RandomForestClassifier | nested AUC 0.995 [0.987, 1.000], accuracy 97.6%
best_rppg_physiology: rPPG v2 physiology [subject-norm] -> LogisticRegression | nested AUC 0.823 [0.732, 0.897], accuracy 78.6%
saved -> /content/drive/MyDrive/UBFC-Phys/processed/results/best_models.joblib


In [20]:
print('=== rPPG quality (vs wrist BVP heart rate) ===')
for r in rec_val.itertuples():
    cov = f' | coverage {100 * r.coverage:.0f}%' if pd.notna(getattr(r, 'coverage', np.nan)) else ''
    print(f'  {r.task} {r.signal:22s} MAE {r.mae:5.1f} BPM | within 5 BPM {100 * r.within5:3.0f}%{cov}')
print('\n=== Rest vs task (nested, leave-subjects-out) ===')
for r in res10.sort_values(['setting', 'auc'], ascending=[True, False]).itertuples():
    print(f'  {r.set:38s} {r.setting:12s} AUC {r.auc:.3f} [{r.auc_lo:.3f}, {r.auc_hi:.3f}] | acc {100 * r.acc:4.1f}% '
          f'| bal.acc {100 * r.bal_acc:4.1f}% | rest identified {100 * r.rest_id:3.0f}%')
print('\n=== Does rPPG add beyond motion? (paired dAUC) ===')
for r in pr.itertuples():
    print(f'  {r.A} vs {r.B} [{r.setting}]: {r.dAUC:+.3f} [{r.lo:+.3f}, {r.hi:+.3f}] -> {r.verdict}')
print('\n=== Stressful vs relaxed version of the same task (nested) ===')
for r in res11.sort_values('auc', ascending=False).itertuples():
    print(f'  {r.set:38s} AUC {r.auc:.3f} [{r.auc_lo:.3f}, {r.auc_hi:.3f}] | acc {100 * r.acc:4.1f}% | bal.acc {100 * r.bal_acc:4.1f}%')

=== rPPG quality (vs wrist BVP heart rate) ===
  T1 old rPPG (part 1)      MAE   7.4 BPM | within 5 BPM  70%
  T1 new rPPG v2 (gated)    MAE   5.9 BPM | within 5 BPM  73% | coverage 85%
  T2 old rPPG (part 1)      MAE  16.9 BPM | within 5 BPM  36%
  T2 new rPPG v2 (gated)    MAE  18.0 BPM | within 5 BPM  32% | coverage 47%
  T3 old rPPG (part 1)      MAE  17.6 BPM | within 5 BPM  32%
  T3 new rPPG v2 (gated)    MAE  17.6 BPM | within 5 BPM  38% | coverage 56%

=== Rest vs task (nested, leave-subjects-out) ===
  rPPG v2 + motion (camera only)         raw          AUC 0.972 [0.944, 0.993] | acc 91.7% | bal.acc 90.6% | rest identified  96%
  motion + quality (shortcut)            raw          AUC 0.961 [0.928, 0.987] | acc 90.5% | bal.acc 89.7% | rest identified  95%
  rPPG v1 physiology (part 1)            raw          AUC 0.898 [0.850, 0.947] | acc 86.3% | bal.acc 83.0% | rest identified  86%
  EDA (wrist)                            raw          AUC 0.873 [0.820, 0.926] | acc 81.5% | ba